In [25]:
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import os
load_dotenv()
key = os.getenv("OPENAI_API_KEY")

In [12]:
from langchain_ollama import OllamaEmbeddings
MODEL = "nomic-embed-text"
embeddings = OllamaEmbeddings(
    model = MODEL,
    base_url= "http://localhost:11434"
)

In [16]:
import tiktoken

vec = embeddings.embed_query("Hi I am Parag")


In [17]:
text = "HI I am Parag"

In [21]:

import tiktoken

MODEL = "nomic-embed-text"

text = "HI I am Parag"

encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode(text)

token_count = len(tokens)

print(f"Total tokens: {token_count}")
print(tokens)

Total tokens: 5
[24860, 358, 1097, 4366, 351]


In [23]:
import glob
knowledge_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_path, recursive = True)
print(f"Number of Files {len(files)}")
entire_knowledge = ""
for file in files:
    with open(file, "r",encoding='utf-8') as f:
        entire_knowledge += f.read()
        entire_knowledge += "\n\n"
print(f"Characters in Entire Knowledge {len(entire_knowledge):,}")



Number of Files 71
Characters in Entire Knowledge 286,212


In [26]:
# Loading the knowledge base as Document type

folder_path = glob.glob("knowledge-base/*")
documents = []

for folder in folder_path:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob = "**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding":"utf-8"})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc)


print(f"Loaded {len(documents)} documents")

Loaded 71 documents


In [27]:
documents[1]

Document(metadata={'source': 'knowledge-base\\contracts\\Contract with Atlantic Risk Solutions for Bizllm.md', 'doc_type': 'contracts'}, page_content="# Contract with Atlantic Risk Solutions for Bizllm\n\n---\n\n## Terms\n\n1. **Agreement Effective Date**: This contract is effective as of January 15, 2025.\n2. **Duration**: This agreement will remain in effect for a term of 12 months, concluding on January 14, 2026.\n3. **Subscription Type**: Atlantic Risk Solutions agrees to subscribe to the **Professional Tier** of Bizllm, at a cost of $12,000/month, totaling $144,000 for the duration of this contract.\n4. **Payment Terms**: Payments are due on the 10th of each month via ACH transfer. Late payments will incur a penalty of 1.5% per month and may result in service suspension after 15 days delinquency.\n5. **User Licenses**: Contract includes 35 named user licenses. Additional users may be added at $180/month per license.\n6. **Termination Clause**: Either party may terminate this agree

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

chunks = splitter.split_documents(documents)

# Metdata is preserved here, splitting takes place on individual Document Object, the page content is splitted

In [30]:
len(chunks)

389

In [34]:
len(chunks[0].page_content)

969

In [36]:
from langchain_chroma import Chroma

db_name = "vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory = db_name, embedding_function = embeddings).delete_collection()

vector_store = Chroma.from_documents(persist_directory=db_name, embedding=embeddings, documents=chunks)